# 10. Modelo MILP emparejado y diferencias XOR

El modelo desarrollado anteriormente representa una ejecución concreta de
Keccak reducido:

$$
A_0
\longrightarrow
A_1
\longrightarrow
\cdots
\longrightarrow
A_R.
$$

Para comenzar el análisis diferencial se construyen ahora dos ejecuciones
independientes:

$$
A_0
\longrightarrow
A_1
\longrightarrow
\cdots
\longrightarrow
A_R,
$$

y:

$$
A'_0
\longrightarrow
A'_1
\longrightarrow
\cdots
\longrightarrow
A'_R.
$$

La diferencia XOR en cada frontera se define como:

$$
\Delta A_r
=
A_r
\oplus
A'_r.
$$

Para cada bit se introduce una variable binaria:

$$
\delta_{r,x,y,k}
=
A_r[x,y,k]
\oplus
A'_r[x,y,k].
$$

La relación XOR se linealiza mediante:

$$
A_r[x,y,k]
+
A'_r[x,y,k]
=
\delta_{r,x,y,k}
+
2q_{r,x,y,k},
$$

donde:

$$
q_{r,x,y,k}\in\{0,1\}.
$$

Este enfoque representa de manera exacta dos ejecuciones concretas y su
diferencia. Todavía no constituye un modelo diferencial abstracto ni
incorpora probabilidades de transición para la capa no lineal `chi`.

In [1]:
# ============================================================
# CONFIGURACIÓN DEL ENTORNO
# ============================================================

from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """Busca la raíz del proyecto."""
    current = start.resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "src").exists():
            return candidate

    raise FileNotFoundError(
        "No se encontró la raíz del proyecto."
    )


PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


print("Raíz del proyecto:", PROJECT_ROOT)
print("Directorio src:", SRC_DIR)

Raíz del proyecto: D:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada
Directorio src: D:\Documentos\000. MSC\3er Ciclo\Cripto\PracticaCalificada\src


In [2]:
# ============================================================
# IMPORTACIONES
# ============================================================

import inspect
import time

import numpy as np

import keccak_milp.layers as layers

from keccak_milp.config import ExperimentConfig
from keccak_milp.differential import (
    PairedKeccakMILPModel,
)


required_methods = [
    "build_paired_model",
    "difference_variable",
    "difference_state_values",
    "concrete_state_values",
    "add_nonzero_input_difference_constraint",
    "set_boundary_difference_weight_objective",
    "set_input_output_difference_objective",
    "solve",
    "objective_value",
]

for method_name in required_methods:
    assert hasattr(
        PairedKeccakMILPModel,
        method_name,
    )


print("API diferencial disponible:")

for method_name in required_methods:
    method = getattr(
        PairedKeccakMILPModel,
        method_name,
    )

    print(
        f" - {method_name}"
        f"{inspect.signature(method)}"
    )

API diferencial disponible:
 - build_paired_model(self) -> 'None'
 - difference_variable(self, boundary_index: 'int', x: 'int', y: 'int', k: 'int') -> 'pulp.LpVariable'
 - difference_state_values(self, boundary_index: 'int', tolerance: 'float' = 0.5) -> 'np.ndarray'
 - concrete_state_values(self, side: 'ExecutionSide', boundary_index: 'int', tolerance: 'float' = 0.5) -> 'np.ndarray'
 - add_nonzero_input_difference_constraint(self) -> 'None'
 - set_boundary_difference_weight_objective(self, boundary_index: 'int') -> 'None'
 - set_input_output_difference_objective(self) -> 'None'
 - solve(self) -> 'str'
 - objective_value(self) -> 'float'


## Interpretación de las dos ejecuciones

El modelo emparejado contiene tres grupos principales de variables:

1. variables de la ejecución izquierda:

   $$
   A_r[x,y,k];
   $$

2. variables de la ejecución derecha:

   $$
   A'_r[x,y,k];
   $$

3. variables de diferencia:

   $$
   \Delta A_r[x,y,k].
   $$

Cada ejecución satisface por separado todas las restricciones de:

$$
\theta,
\quad
\rho,
\quad
\pi,
\quad
\chi,
\quad
\iota.
$$

La diferencia no se propaga mediante una aproximación independiente. Se
calcula exactamente a partir de los dos estados concretos.

Por tanto:

$$
\Delta A_r
=
A_r
\oplus
A'_r
$$

se cumple en todas las fronteras del modelo.

In [3]:
# ============================================================
# FUNCIONES AUXILIARES
# ============================================================

def hamming_weight(
    state: np.ndarray,
) -> int:
    """Calcula el peso de Hamming de un estado binario."""
    return int(
        np.asarray(
            state,
            dtype=np.int64,
        ).sum()
    )


def active_positions(
    state: np.ndarray,
) -> list[tuple[int, int, int]]:
    """Devuelve las posiciones activas."""
    array = np.asarray(
        state,
        dtype=np.int64,
    )

    return [
        (x, y, k)
        for x in range(array.shape[0])
        for y in range(array.shape[1])
        for k in range(array.shape[2])
        if array[x, y, k] == 1
    ]


def lane_activity(
    state: np.ndarray,
) -> np.ndarray:
    """
    Cuenta los bits activos de cada lane.

    La salida tiene forma:

        (5, 5)
    """
    array = np.asarray(
        state,
        dtype=np.int64,
    )

    return array.sum(axis=2)


def solve_and_measure(
    model: PairedKeccakMILPModel,
) -> tuple[str, float]:
    """Resuelve el modelo y mide el tiempo."""
    start = time.perf_counter()

    status = model.solve()

    elapsed = time.perf_counter() - start

    return status, elapsed


def fix_input_pair(
    model: PairedKeccakMILPModel,
    left_state: np.ndarray,
    right_state: np.ndarray,
    constraint_prefix: str,
) -> None:
    """Fija completamente los dos estados iniciales."""
    left_array = np.asarray(
        left_state,
        dtype=np.int64,
    )

    right_array = np.asarray(
        right_state,
        dtype=np.int64,
    )

    expected_shape = (
        5,
        5,
        model.config.z,
    )

    if left_array.shape != expected_shape:
        raise ValueError(
            "La entrada izquierda debe tener forma "
            f"{expected_shape}."
        )

    if right_array.shape != expected_shape:
        raise ValueError(
            "La entrada derecha debe tener forma "
            f"{expected_shape}."
        )

    if not np.all(
        np.isin(left_array, [0, 1])
    ):
        raise ValueError(
            "La entrada izquierda debe ser binaria."
        )

    if not np.all(
        np.isin(right_array, [0, 1])
    ):
        raise ValueError(
            "La entrada derecha debe ser binaria."
        )

    for x in range(5):
        for y in range(5):
            for k in range(model.config.z):
                model.problem += (
                    model.left.state_variable(
                        0,
                        x,
                        y,
                        k,
                    )
                    == int(left_array[x, y, k]),
                    (
                        f"{constraint_prefix}"
                        f"_left_x{x}_y{y}_k{k}"
                    ),
                )

                model.problem += (
                    model.right.state_variable(
                        0,
                        x,
                        y,
                        k,
                    )
                    == int(right_array[x, y, k]),
                    (
                        f"{constraint_prefix}"
                        f"_right_x{x}_y{y}_k{k}"
                    ),
                )


print("Funciones auxiliares definidas correctamente.")

Funciones auxiliares definidas correctamente.


## Experimento 1: diferencia inicial controlada

Se construirá una ejecución izquierda aleatoria:

$$
A_0,
$$

y una ejecución derecha obtenida al activar una diferencia conocida:

$$
A'_0
=
A_0
\oplus
\Delta A_0.
$$

La diferencia inicial tendrá tres bits activos:

$$
\operatorname{HW}(\Delta A_0)=3.
$$

Después de una ronda se verificará que:

$$
\Delta A_1
=
A_1
\oplus
A'_1.
$$

También se compararán ambas ejecuciones MILP con la implementación de
referencia.

In [4]:
# ============================================================
# EXPERIMENTO 1: PAR DE ENTRADAS CONTROLADO
# ============================================================

z = 4
rounds = 1

rng = np.random.default_rng(2026)

left_input = rng.integers(
    0,
    2,
    size=(5, 5, z),
    dtype=np.int64,
)

controlled_input_difference = np.zeros(
    (5, 5, z),
    dtype=np.int64,
)

controlled_active_bits = [
    (0, 0, 0),
    (1, 2, 1),
    (4, 3, 2),
]

for x, y, k in controlled_active_bits:
    controlled_input_difference[x, y, k] = 1


right_input = np.bitwise_xor(
    left_input,
    controlled_input_difference,
)


print(
    "Peso de A_0:",
    hamming_weight(left_input),
)

print(
    "Peso de A'_0:",
    hamming_weight(right_input),
)

print(
    "Peso de Delta A_0:",
    hamming_weight(controlled_input_difference),
)

print(
    "Bits activos de Delta A_0:",
    active_positions(controlled_input_difference),
)


assert hamming_weight(
    controlled_input_difference
) == len(controlled_active_bits)

assert np.array_equal(
    np.bitwise_xor(
        left_input,
        right_input,
    ),
    controlled_input_difference,
)

Peso de A_0: 47
Peso de A'_0: 44
Peso de Delta A_0: 3
Bits activos de Delta A_0: [(0, 0, 0), (1, 2, 1), (4, 3, 2)]


In [5]:
# ============================================================
# CONSTRUCCIÓN DEL MODELO EMPAREJADO
# ============================================================

paired_config = ExperimentConfig(
    z=z,
    rounds=rounds,
    solver="cbc",
    time_limit_seconds=60,
    verbose=False,
)

paired_model = PairedKeccakMILPModel(
    paired_config
)

paired_model.build_paired_model()


print(
    "Variables declaradas:",
    paired_model.declared_variable_count(),
)

print(
    "Variables conectadas:",
    paired_model.attached_variable_count(),
)

print(
    "Restricciones:",
    paired_model.constraint_count(),
)


expected_single_variables = (
    25 * z
    + 195 * z * rounds
)

expected_difference_variables = (
    2
    * (rounds + 1)
    * 25
    * z
)

expected_total_variables = (
    2 * expected_single_variables
    + expected_difference_variables
)

expected_total_constraints = (
    2 * 185 * z * rounds
    + (rounds + 1) * 25 * z
)


print(
    "Variables esperadas:",
    expected_total_variables,
)

print(
    "Restricciones esperadas:",
    expected_total_constraints,
)


assert (
    paired_model.declared_variable_count()
    == expected_total_variables
)

assert (
    paired_model.constraint_count()
    == expected_total_constraints
)

Variables declaradas: 2160
Variables conectadas: 2160
Restricciones: 1680
Variables esperadas: 2160
Restricciones esperadas: 1680


In [13]:
# ============================================================
# RECONSTRUIR, FIJAR LAS DOS ENTRADAS Y RESOLVER
# ============================================================

# Se crea un modelo nuevo para que la celda pueda ejecutarse
# nuevamente sin duplicar nombres de restricciones.
paired_model = PairedKeccakMILPModel(
    paired_config
)

paired_model.build_paired_model()

fix_input_pair(
    model=paired_model,
    left_state=left_input,
    right_state=right_input,
    constraint_prefix="experiment_1",
)

paired_model.set_boundary_difference_weight_objective(
    boundary_index=0
)

paired_status, paired_time = solve_and_measure(
    paired_model
)


print("Estado del solver:", paired_status)

print(
    "Tiempo:",
    f"{paired_time:.4f} segundos",
)

print(
    "Valor objetivo:",
    paired_model.objective_value(),
)


assert paired_status == "Optimal"

assert (
    paired_model.objective_value()
    == float(
        hamming_weight(
            controlled_input_difference
        )
    )
)

Estado del solver: Optimal
Tiempo: 0.3466 segundos
Valor objetivo: 3.0


In [14]:
# ============================================================
# RECUPERACIÓN DE ESTADOS Y DIFERENCIAS
# ============================================================

milp_left_input = paired_model.concrete_state_values(
    side="left",
    boundary_index=0,
)

milp_right_input = paired_model.concrete_state_values(
    side="right",
    boundary_index=0,
)

milp_input_difference = (
    paired_model.difference_state_values(
        boundary_index=0
    )
)

milp_left_output = paired_model.concrete_state_values(
    side="left",
    boundary_index=1,
)

milp_right_output = paired_model.concrete_state_values(
    side="right",
    boundary_index=1,
)

milp_output_difference = (
    paired_model.difference_state_values(
        boundary_index=1
    )
)


print(
    "HW(Delta A_0):",
    hamming_weight(milp_input_difference),
)

print(
    "HW(Delta A_1):",
    hamming_weight(milp_output_difference),
)

print(
    "Bits activos de Delta A_0:",
    active_positions(milp_input_difference),
)

print(
    "Número de bits activos de Delta A_1:",
    len(
        active_positions(
            milp_output_difference
        )
    ),
)


assert np.array_equal(
    milp_left_input,
    left_input,
)

assert np.array_equal(
    milp_right_input,
    right_input,
)

assert np.array_equal(
    milp_input_difference,
    controlled_input_difference,
)

assert np.array_equal(
    milp_input_difference,
    np.bitwise_xor(
        milp_left_input,
        milp_right_input,
    ),
)

assert np.array_equal(
    milp_output_difference,
    np.bitwise_xor(
        milp_left_output,
        milp_right_output,
    ),
)

HW(Delta A_0): 3
HW(Delta A_1): 35
Bits activos de Delta A_0: [(0, 0, 0), (1, 2, 1), (4, 3, 2)]
Número de bits activos de Delta A_1: 35


In [15]:
# ============================================================
# COMPARACIÓN CON LA IMPLEMENTACIÓN DE REFERENCIA
# ============================================================

reference_left_output = layers.keccak_rounds(
    left_input,
    number_of_rounds=rounds,
)

reference_right_output = layers.keccak_rounds(
    right_input,
    number_of_rounds=rounds,
)

reference_output_difference = np.bitwise_xor(
    reference_left_output,
    reference_right_output,
)


left_output_differences = int(
    np.count_nonzero(
        milp_left_output
        != reference_left_output
    )
)

right_output_differences = int(
    np.count_nonzero(
        milp_right_output
        != reference_right_output
    )
)

xor_output_differences = int(
    np.count_nonzero(
        milp_output_difference
        != reference_output_difference
    )
)


print(
    "Diferencias en la ejecución izquierda:",
    left_output_differences,
)

print(
    "Diferencias en la ejecución derecha:",
    right_output_differences,
)

print(
    "Diferencias en Delta A_1:",
    xor_output_differences,
)


assert left_output_differences == 0
assert right_output_differences == 0
assert xor_output_differences == 0

Diferencias en la ejecución izquierda: 0
Diferencias en la ejecución derecha: 0
Diferencias en Delta A_1: 0


In [16]:
# ============================================================
# ACTIVIDAD DIFERENCIAL POR LANE
# ============================================================

input_lane_activity = lane_activity(
    milp_input_difference
)

output_lane_activity = lane_activity(
    milp_output_difference
)


print("Actividad por lane en Delta A_0:")
print(input_lane_activity)

print()

print("Actividad por lane en Delta A_1:")
print(output_lane_activity)


assert int(
    input_lane_activity.sum()
) == hamming_weight(
    milp_input_difference
)

assert int(
    output_lane_activity.sum()
) == hamming_weight(
    milp_output_difference
)

Actividad por lane en Delta A_0:
[[1 0 0 0 0]
 [0 0 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 1 0]]

Actividad por lane en Delta A_1:
[[0 2 1 2 2]
 [1 1 3 1 2]
 [1 0 2 2 1]
 [0 2 2 2 1]
 [2 2 0 2 1]]


### Interpretación del primer experimento

El modelo debe satisfacer simultáneamente:

$$
A_1
=
\operatorname{KeccakRound}(A_0),
$$

$$
A'_1
=
\operatorname{KeccakRound}(A'_0),
$$

y:

$$
\Delta A_1
=
A_1
\oplus
A'_1.
$$

La comparación con la referencia debe producir:

```text
Diferencias en la ejecución izquierda: 0
Diferencias en la ejecución derecha: 0
Diferencias en Delta A_1: 0

## Experimento 2: propagación diferencial durante dos rondas

Ahora se utilizará el mismo par de entradas y se construirán dos rondas:

$$
A_0
\longrightarrow
A_1
\longrightarrow
A_2,
$$

$$
A'_0
\longrightarrow
A'_1
\longrightarrow
A'_2.
$$

En cada frontera se calculará:

$$
\Delta A_r
=
A_r
\oplus
A'_r.
$$

El experimento permitirá comparar:

$$
\operatorname{HW}(\Delta A_0),
\qquad
\operatorname{HW}(\Delta A_1),
\qquad
\operatorname{HW}(\Delta A_2).
$$

Como las dos entradas están completamente fijadas, el modelo representa una
propagación determinista y debe resolverse rápidamente.

In [17]:
# ============================================================
# EXPERIMENTO 2: MODELO EMPAREJADO DE DOS RONDAS
# ============================================================

two_round_config = ExperimentConfig(
    z=z,
    rounds=2,
    solver="cbc",
    time_limit_seconds=60,
    verbose=False,
)

two_round_model = PairedKeccakMILPModel(
    two_round_config
)

two_round_model.build_paired_model()

fix_input_pair(
    model=two_round_model,
    left_state=left_input,
    right_state=right_input,
    constraint_prefix="experiment_2",
)

# El objetivo se fija sobre la diferencia inicial.
# Como las entradas están fijadas, no altera la propagación.
two_round_model.set_boundary_difference_weight_objective(
    boundary_index=0
)

two_round_status, two_round_time = solve_and_measure(
    two_round_model
)


print("Estado del solver:", two_round_status)

print(
    "Tiempo:",
    f"{two_round_time:.4f} segundos",
)

print(
    "Variables declaradas:",
    two_round_model.declared_variable_count(),
)

print(
    "Variables conectadas:",
    two_round_model.attached_variable_count(),
)

print(
    "Restricciones:",
    two_round_model.constraint_count(),
)

print(
    "Valor objetivo:",
    two_round_model.objective_value(),
)


assert two_round_status == "Optimal"

assert two_round_model.objective_value() == float(
    hamming_weight(
        controlled_input_difference
    )
)

Estado del solver: Optimal
Tiempo: 0.3645 segundos
Variables declaradas: 3920
Variables conectadas: 3920
Restricciones: 3460
Valor objetivo: 3.0


In [18]:
# ============================================================
# RECUPERAR DIFERENCIAS EN A_0, A_1 Y A_2
# ============================================================

two_round_differences = []

for boundary_index in range(3):
    difference = (
        two_round_model.difference_state_values(
            boundary_index=boundary_index
        )
    )

    two_round_differences.append(
        difference
    )


delta_a0 = two_round_differences[0]
delta_a1 = two_round_differences[1]
delta_a2 = two_round_differences[2]


weight_delta_a0 = hamming_weight(delta_a0)
weight_delta_a1 = hamming_weight(delta_a1)
weight_delta_a2 = hamming_weight(delta_a2)


print("HW(Delta A_0):", weight_delta_a0)
print("HW(Delta A_1):", weight_delta_a1)
print("HW(Delta A_2):", weight_delta_a2)


assert weight_delta_a0 == 3

# Debe coincidir con el resultado del primer experimento.
assert np.array_equal(
    delta_a1,
    milp_output_difference,
)

HW(Delta A_0): 3
HW(Delta A_1): 35
HW(Delta A_2): 49


In [19]:
# ============================================================
# VERIFICACIÓN DE LAS DOS RONDAS CON LA REFERENCIA
# ============================================================

reference_left_a1 = layers.keccak_round(
    left_input,
    round_index=0,
)

reference_right_a1 = layers.keccak_round(
    right_input,
    round_index=0,
)

reference_left_a2 = layers.keccak_round(
    reference_left_a1,
    round_index=1,
)

reference_right_a2 = layers.keccak_round(
    reference_right_a1,
    round_index=1,
)


reference_delta_a0 = np.bitwise_xor(
    left_input,
    right_input,
)

reference_delta_a1 = np.bitwise_xor(
    reference_left_a1,
    reference_right_a1,
)

reference_delta_a2 = np.bitwise_xor(
    reference_left_a2,
    reference_right_a2,
)


difference_counts = {
    "Delta A_0": int(
        np.count_nonzero(
            delta_a0 != reference_delta_a0
        )
    ),
    "Delta A_1": int(
        np.count_nonzero(
            delta_a1 != reference_delta_a1
        )
    ),
    "Delta A_2": int(
        np.count_nonzero(
            delta_a2 != reference_delta_a2
        )
    ),
}


for name, count in difference_counts.items():
    print(
        f"Diferencias en {name}:",
        count,
    )


assert all(
    count == 0
    for count in difference_counts.values()
)

Diferencias en Delta A_0: 0
Diferencias en Delta A_1: 0
Diferencias en Delta A_2: 0


In [20]:
# ============================================================
# ACTIVIDAD DIFERENCIAL POR LANE
# ============================================================

lane_activity_a0 = lane_activity(delta_a0)
lane_activity_a1 = lane_activity(delta_a1)
lane_activity_a2 = lane_activity(delta_a2)


print("Actividad por lane en Delta A_0:")
print(lane_activity_a0)

print()

print("Actividad por lane en Delta A_1:")
print(lane_activity_a1)

print()

print("Actividad por lane en Delta A_2:")
print(lane_activity_a2)

Actividad por lane en Delta A_0:
[[1 0 0 0 0]
 [0 0 1 0 0]
 [0 0 0 0 0]
 [0 0 0 0 0]
 [0 0 0 1 0]]

Actividad por lane en Delta A_1:
[[0 2 1 2 2]
 [1 1 3 1 2]
 [1 0 2 2 1]
 [0 2 2 2 1]
 [2 2 0 2 1]]

Actividad por lane en Delta A_2:
[[3 2 1 1 3]
 [1 2 3 4 3]
 [3 1 1 2 2]
 [4 1 3 2 2]
 [0 1 2 1 1]]


In [21]:
# ============================================================
# RESUMEN CUANTITATIVO DE LA PROPAGACIÓN
# ============================================================

propagation_summary = []

for boundary_index, difference in enumerate(
    two_round_differences
):
    activity = lane_activity(
        difference
    )

    bit_weight = hamming_weight(
        difference
    )

    active_lane_count = int(
        np.count_nonzero(activity)
    )

    propagation_summary.append(
        {
            "frontera": f"Delta A_{boundary_index}",
            "bits_activos": bit_weight,
            "lanes_activos": active_lane_count,
            "densidad_bits": (
                bit_weight
                / difference.size
            ),
            "cobertura_lanes": (
                active_lane_count
                / 25
            ),
        }
    )


header = (
    "Frontera  | Bits activos | Lanes activos | "
    "Densidad | Cobertura"
)

print(header)
print("-" * len(header))

for result in propagation_summary:
    print(
        f"{result['frontera']:<9} | "
        f"{result['bits_activos']:>12} | "
        f"{result['lanes_activos']:>13} | "
        f"{result['densidad_bits']:>7.2%} | "
        f"{result['cobertura_lanes']:>8.2%}"
    )

Frontera  | Bits activos | Lanes activos | Densidad | Cobertura
---------------------------------------------------------------
Delta A_0 |            3 |             3 |   3.00% |   12.00%
Delta A_1 |           35 |            21 |  35.00% |   84.00%
Delta A_2 |           49 |            24 |  49.00% |   96.00%


### Interpretación de la propagación en dos rondas

El peso de Hamming permite medir cuántas posiciones binarias son diferentes:

$$
\operatorname{HW}(\Delta A_r).
$$

La cantidad de lanes activos mide cuántas posiciones espaciales contienen al
menos un bit diferente:

$$
N_{\mathrm{lanes}}(\Delta A_r)
=
\sum_{x,y}
\mathbf{1}
\left[
\sum_k \Delta A_r[x,y,k]>0
\right].
$$

Ambas métricas describen aspectos distintos:

- el peso de bits cuantifica la cantidad total de diferencias;
- la actividad por lane cuantifica la distribución espacial;
- la densidad mide la proporción del estado que está activa;
- la cobertura indica qué fracción de los 25 lanes contiene diferencias.

Una ronda ya distribuyó tres bits iniciales sobre 21 lanes. La segunda ronda
permitirá comprobar si la actividad alcanza la totalidad o casi la totalidad
del estado reducido.

El resultado sigue correspondiendo a un par concreto:

$$
(A_0,A'_0).
$$

No se puede concluir todavía que todas las diferencias de peso tres sigan el
mismo patrón.

## Experimento 3: influencia del estado base

En una función lineal, una diferencia de entrada determina completamente la
diferencia de salida:

$$
F(A)\oplus F(A\oplus\Delta A)
$$

sería independiente de $A$.

Sin embargo, Keccak contiene la capa no lineal `chi`. Por ello, la salida
diferencial puede depender del estado concreto sobre el cual se aplica la
diferencia.

Se mantendrá fija:

$$
\Delta A_0,
$$

pero se generarán distintos estados base:

$$
A_0^{(1)},A_0^{(2)},\ldots,A_0^{(n)}.
$$

Para cada uno se calculará:

$$
A'_0=A_0\oplus\Delta A_0,
$$

y se medirán:

$$
\operatorname{HW}(\Delta A_1)
\quad\text{y}\quad
\operatorname{HW}(\Delta A_2).
$$

Este experimento se realizará primero con la implementación de referencia,
porque permite analizar múltiples pares con bajo costo computacional.

In [22]:
# ============================================================
# PROPAGACIÓN DIFERENCIAL MEDIANTE LA REFERENCIA
# ============================================================

def propagate_pair_reference(
    left_state: np.ndarray,
    input_difference: np.ndarray,
    number_of_rounds: int,
) -> list[np.ndarray]:
    """
    Propaga un par de estados y devuelve las diferencias
    en todas las fronteras.
    """
    left_current = np.asarray(
        left_state,
        dtype=np.int64,
    ).copy()

    input_delta = np.asarray(
        input_difference,
        dtype=np.int64,
    )

    right_current = np.bitwise_xor(
        left_current,
        input_delta,
    )

    differences = [
        np.bitwise_xor(
            left_current,
            right_current,
        )
    ]

    for round_index in range(number_of_rounds):
        left_current = layers.keccak_round(
            left_current,
            round_index=round_index,
        )

        right_current = layers.keccak_round(
            right_current,
            round_index=round_index,
        )

        differences.append(
            np.bitwise_xor(
                left_current,
                right_current,
            )
        )

    return differences


reference_test_differences = propagate_pair_reference(
    left_state=left_input,
    input_difference=controlled_input_difference,
    number_of_rounds=2,
)


assert np.array_equal(
    reference_test_differences[0],
    delta_a0,
)

assert np.array_equal(
    reference_test_differences[1],
    delta_a1,
)

assert np.array_equal(
    reference_test_differences[2],
    delta_a2,
)


print(
    "Función de propagación diferencial validada."
)

Función de propagación diferencial validada.


In [23]:
# ============================================================
# MISMA DIFERENCIA, DISTINTOS ESTADOS BASE
# ============================================================

number_of_samples = 20
sample_results = []

for sample_index in range(number_of_samples):
    sample_rng = np.random.default_rng(
        10_000 + sample_index
    )

    sample_left_input = sample_rng.integers(
        0,
        2,
        size=(5, 5, z),
        dtype=np.int64,
    )

    sample_differences = propagate_pair_reference(
        left_state=sample_left_input,
        input_difference=controlled_input_difference,
        number_of_rounds=2,
    )

    sample_results.append(
        {
            "muestra": sample_index,
            "peso_delta_a0": hamming_weight(
                sample_differences[0]
            ),
            "peso_delta_a1": hamming_weight(
                sample_differences[1]
            ),
            "peso_delta_a2": hamming_weight(
                sample_differences[2]
            ),
            "lanes_delta_a1": int(
                np.count_nonzero(
                    lane_activity(
                        sample_differences[1]
                    )
                )
            ),
            "lanes_delta_a2": int(
                np.count_nonzero(
                    lane_activity(
                        sample_differences[2]
                    )
                )
            ),
        }
    )


header = (
    "Muestra | HW(Delta A_0) | HW(Delta A_1) | "
    "HW(Delta A_2) | Lanes A_1 | Lanes A_2"
)

print(header)
print("-" * len(header))

for result in sample_results:
    print(
        f"{result['muestra']:>7} | "
        f"{result['peso_delta_a0']:>13} | "
        f"{result['peso_delta_a1']:>13} | "
        f"{result['peso_delta_a2']:>13} | "
        f"{result['lanes_delta_a1']:>9} | "
        f"{result['lanes_delta_a2']:>9}"
    )


assert all(
    result["peso_delta_a0"] == 3
    for result in sample_results
)

Muestra | HW(Delta A_0) | HW(Delta A_1) | HW(Delta A_2) | Lanes A_1 | Lanes A_2
-------------------------------------------------------------------------------
      0 |             3 |            32 |            54 |        21 |        25
      1 |             3 |            35 |            46 |        24 |        22
      2 |             3 |            36 |            53 |        22 |        23
      3 |             3 |            35 |            54 |        20 |        24
      4 |             3 |            37 |            55 |        18 |        24
      5 |             3 |            38 |            45 |        21 |        24
      6 |             3 |            33 |            49 |        19 |        24
      7 |             3 |            40 |            49 |        23 |        23
      8 |             3 |            32 |            52 |        19 |        25
      9 |             3 |            31 |            50 |        20 |        24
     10 |             3 |            40 

In [24]:
# ============================================================
# RESUMEN ESTADÍSTICO
# ============================================================

weights_a1 = np.array(
    [
        result["peso_delta_a1"]
        for result in sample_results
    ],
    dtype=np.int64,
)

weights_a2 = np.array(
    [
        result["peso_delta_a2"]
        for result in sample_results
    ],
    dtype=np.int64,
)

lanes_a1 = np.array(
    [
        result["lanes_delta_a1"]
        for result in sample_results
    ],
    dtype=np.int64,
)

lanes_a2 = np.array(
    [
        result["lanes_delta_a2"]
        for result in sample_results
    ],
    dtype=np.int64,
)


def print_summary(
    name: str,
    values: np.ndarray,
) -> None:
    """Imprime un resumen descriptivo sencillo."""
    print(name)

    print(
        "  mínimo:",
        int(values.min()),
    )

    print(
        "  máximo:",
        int(values.max()),
    )

    print(
        "  media:",
        f"{values.mean():.2f}",
    )

    print(
        "  desviación estándar:",
        f"{values.std(ddof=1):.2f}",
    )


print_summary(
    "Peso de Delta A_1",
    weights_a1,
)

print()

print_summary(
    "Peso de Delta A_2",
    weights_a2,
)

print()

print_summary(
    "Lanes activos en Delta A_1",
    lanes_a1,
)

print()

print_summary(
    "Lanes activos en Delta A_2",
    lanes_a2,
)

Peso de Delta A_1
  mínimo: 31
  máximo: 40
  media: 35.65
  desviación estándar: 2.78

Peso de Delta A_2
  mínimo: 43
  máximo: 60
  media: 51.20
  desviación estándar: 4.67

Lanes activos en Delta A_1
  mínimo: 18
  máximo: 24
  media: 21.00
  desviación estándar: 1.65

Lanes activos en Delta A_2
  mínimo: 22
  máximo: 25
  media: 23.75
  desviación estándar: 0.91


### Interpretación de la variabilidad

La diferencia inicial permanece fija:

$$
\operatorname{HW}(\Delta A_0)=3.
$$

Sin embargo, los pesos de las diferencias posteriores pueden cambiar entre
muestras:

$$
\operatorname{HW}(\Delta A_1)
\neq
\text{constante},
$$

$$
\operatorname{HW}(\Delta A_2)
\neq
\text{constante}.
$$

Esto ocurre porque la capa `chi` es no lineal. La propagación diferencial no
depende solamente de la diferencia de entrada, sino también de los valores
concretos de los bits involucrados.

En consecuencia, una misma diferencia:

$$
\Delta A_0
$$

puede producir varias diferencias de salida posibles.

El modelo emparejado representa una realización concreta de esta transición.
Un modelo diferencial abstracto deberá describir el conjunto de transiciones
posibles y, posteriormente, su costo o probabilidad.

In [25]:
# ============================================================
# VALIDACIÓN MILP DE UNA MUESTRA ADICIONAL
# ============================================================

validation_sample_index = 5
validation_seed = 10_000 + validation_sample_index

validation_rng = np.random.default_rng(
    validation_seed
)

validation_left_input = validation_rng.integers(
    0,
    2,
    size=(5, 5, z),
    dtype=np.int64,
)

validation_right_input = np.bitwise_xor(
    validation_left_input,
    controlled_input_difference,
)

validation_reference = propagate_pair_reference(
    left_state=validation_left_input,
    input_difference=controlled_input_difference,
    number_of_rounds=2,
)


validation_config = ExperimentConfig(
    z=z,
    rounds=2,
    solver="cbc",
    time_limit_seconds=60,
    verbose=False,
)

validation_model = PairedKeccakMILPModel(
    validation_config
)

validation_model.build_paired_model()

fix_input_pair(
    model=validation_model,
    left_state=validation_left_input,
    right_state=validation_right_input,
    constraint_prefix="experiment_3_validation",
)

validation_model.set_boundary_difference_weight_objective(
    boundary_index=0
)

validation_status, validation_time = solve_and_measure(
    validation_model
)


assert validation_status == "Optimal"


validation_milp_differences = [
    validation_model.difference_state_values(
        boundary_index=boundary_index
    )
    for boundary_index in range(3)
]


validation_error_counts = []

for boundary_index in range(3):
    error_count = int(
        np.count_nonzero(
            validation_milp_differences[boundary_index]
            != validation_reference[boundary_index]
        )
    )

    validation_error_counts.append(
        error_count
    )

    print(
        f"Diferencias en Delta A_{boundary_index}:",
        error_count,
    )


print(
    "Tiempo de validación MILP:",
    f"{validation_time:.4f} segundos",
)


assert validation_error_counts == [0, 0, 0]

Diferencias en Delta A_0: 0
Diferencias en Delta A_1: 0
Diferencias en Delta A_2: 0
Tiempo de validación MILP: 0.3694 segundos


## Comparación de los patrones diferenciales

Los resultados anteriores compararon únicamente los pesos de Hamming.

Sin embargo, dos diferencias pueden tener el mismo peso y ocupar posiciones
completamente distintas. Por ello, también se comprobará cuántos patrones
diferenciales distintos aparecen para una misma diferencia inicial:

$$
\Delta A_0.
$$

Debido a la no linealidad de `chi`, se espera que distintos estados base
produzcan diferentes configuraciones de:

$$
\Delta A_1
\quad\text{y}\quad
\Delta A_2.
$$

In [26]:
# ============================================================
# CONTEO DE PATRONES DIFERENCIALES DISTINTOS
# ============================================================

difference_patterns_a1 = []
difference_patterns_a2 = []

for sample_index in range(number_of_samples):
    sample_rng = np.random.default_rng(
        10_000 + sample_index
    )

    sample_left_input = sample_rng.integers(
        0,
        2,
        size=(5, 5, z),
        dtype=np.int64,
    )

    sample_differences = propagate_pair_reference(
        left_state=sample_left_input,
        input_difference=controlled_input_difference,
        number_of_rounds=2,
    )

    difference_patterns_a1.append(
        sample_differences[1]
    )

    difference_patterns_a2.append(
        sample_differences[2]
    )


unique_patterns_a1 = {
    difference.tobytes()
    for difference in difference_patterns_a1
}

unique_patterns_a2 = {
    difference.tobytes()
    for difference in difference_patterns_a2
}


print(
    "Número de muestras:",
    number_of_samples,
)

print(
    "Patrones distintos en Delta A_1:",
    len(unique_patterns_a1),
)

print(
    "Patrones distintos en Delta A_2:",
    len(unique_patterns_a2),
)


assert len(unique_patterns_a1) > 1
assert len(unique_patterns_a2) > 1

Número de muestras: 20
Patrones distintos en Delta A_1: 20
Patrones distintos en Delta A_2: 20


## Experimento 4: diferencia inicial nula

Como prueba de consistencia se fijarán dos entradas idénticas:

$$
A'_0=A_0.
$$

Por definición:

$$
\Delta A_0
=
A_0\oplus A'_0
=
0.
$$

Como ambas ejecuciones aplican exactamente las mismas rondas y constantes:

$$
A_r=A'_r
$$

para toda frontera $r$. En consecuencia:

$$
\Delta A_r=0.
$$

Este experimento comprueba que el modelo no introduce diferencias
artificiales.

In [27]:
# ============================================================
# EXPERIMENTO 4: DIFERENCIA NULA
# ============================================================

zero_difference_rng = np.random.default_rng(
    70_126
)

identical_input = zero_difference_rng.integers(
    0,
    2,
    size=(5, 5, z),
    dtype=np.int64,
)

zero_difference_config = ExperimentConfig(
    z=z,
    rounds=2,
    solver="cbc",
    time_limit_seconds=60,
    verbose=False,
)

zero_difference_model = PairedKeccakMILPModel(
    zero_difference_config
)

zero_difference_model.build_paired_model()

fix_input_pair(
    model=zero_difference_model,
    left_state=identical_input,
    right_state=identical_input.copy(),
    constraint_prefix="zero_difference",
)

zero_difference_model.set_input_output_difference_objective()

(
    zero_difference_status,
    zero_difference_time,
) = solve_and_measure(
    zero_difference_model
)


print(
    "Estado del solver:",
    zero_difference_status,
)

print(
    "Tiempo:",
    f"{zero_difference_time:.4f} segundos",
)

print(
    "Valor objetivo:",
    zero_difference_model.objective_value(),
)


assert zero_difference_status == "Optimal"
assert zero_difference_model.objective_value() == 0.0

Estado del solver: Optimal
Tiempo: 0.3692 segundos
Valor objetivo: 0.0


In [28]:
# ============================================================
# COMPROBACIÓN DE LA DIFERENCIA NULA
# ============================================================

zero_difference_weights = []

for boundary_index in range(3):
    difference = (
        zero_difference_model.difference_state_values(
            boundary_index=boundary_index
        )
    )

    weight = hamming_weight(
        difference
    )

    zero_difference_weights.append(
        weight
    )

    print(
        f"HW(Delta A_{boundary_index}):",
        weight,
    )


assert zero_difference_weights == [0, 0, 0]

HW(Delta A_0): 0
HW(Delta A_1): 0
HW(Delta A_2): 0


## Modelo emparejado exacto y modelo diferencial abstracto

Es importante diferenciar dos enfoques.

### Modelo emparejado exacto

El modelo desarrollado en este notebook contiene dos ejecuciones concretas:

$$
A_r
\quad\text{y}\quad
A'_r.
$$

La diferencia se obtiene mediante:

$$
\Delta A_r
=
A_r\oplus A'_r.
$$

Por tanto, el modelo responde preguntas como:

> ¿Qué diferencia se obtiene para este par concreto de entradas?

La transición queda completamente determinada una vez que se fijan ambos
estados iniciales.

### Modelo diferencial abstracto

Un modelo diferencial abstracto debería trabajar directamente con:

$$
\Delta A_r,
$$

sin representar necesariamente los valores completos de:

$$
A_r
\quad\text{y}\quad
A'_r.
$$

Este modelo debería responder preguntas como:

> ¿Qué diferencias de salida son compatibles con una diferencia de entrada?

Para las capas lineales, la diferencia puede propagarse directamente. Sin
embargo, para la capa no lineal `chi` deben representarse las transiciones
diferenciales posibles.

También será necesario distinguir entre:

- existencia de una transición;
- número de pares que producen esa transición;
- probabilidad o peso diferencial;
- número de S-boxes o filas no lineales activas.

El modelo emparejado exacto sirve como referencia para validar posteriormente
esa formulación abstracta.

## Conclusiones

El modelo emparejado permitió representar dos ejecuciones completas de
Keccak reducido y definir diferencias XOR exactas en todas sus fronteras.

Los resultados principales son:

1. Para cada frontera se cumple:

   $$
   \Delta A_r
   =
   A_r\oplus A'_r.
   $$

2. La linealización:

   $$
   A_r[x,y,k]+A'_r[x,y,k]
   =
   \delta_{r,x,y,k}
   +
   2q_{r,x,y,k}
   $$

   representa correctamente la operación XOR.

3. Las ejecuciones izquierda y derecha coinciden con la implementación de
   referencia.

4. Para el par inicial analizado se obtuvo:

   $$
   \operatorname{HW}(\Delta A_0)=3,
   $$

   $$
   \operatorname{HW}(\Delta A_1)=35,
   $$

   $$
   \operatorname{HW}(\Delta A_2)=49.
   $$

5. La cobertura de lanes evolucionó de:

   $$
   12\%
   \longrightarrow
   84\%
   \longrightarrow
   96\%.
   $$

6. Al mantener fija la diferencia inicial y modificar el estado base, el peso
   de la diferencia después de una ronda varió entre 31 y 40.

7. Después de dos rondas, el peso diferencial varió entre 43 y 60.

8. La cobertura media después de dos rondas fue de 23.75 lanes, equivalente
   al 95% del estado.

9. La misma diferencia inicial produjo varios patrones de salida distintos,
   lo que evidencia la dependencia respecto del estado base introducida por
   `chi`.

10. Cuando las dos entradas son iguales, todas las diferencias permanecen
    nulas.

El modelo actual describe realizaciones diferenciales concretas. La siguiente
etapa consistirá en formular la propagación diferencial directamente sobre
las variables de diferencia y analizar la actividad no lineal de `chi`.